In [1]:
# Librerías a instalar
#pip install langdetect deep-translator pandas

In [2]:
# Librerías
from langdetect import detect, LangDetectException
from deep_translator import GoogleTranslator
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import os

## DETECTAR EL IDIOMA Y TRADUCIR

In [3]:
def detectar_idioma(texto):
    try:
        if pd.isna(texto) or str(texto).strip() == '': # si es un NA o sólo espacios vacíos no se traduce
            return 'es'  
        return detect(str(texto)) # devuelve un string con el idioma en el que está
    except LangDetectException:
        return 'desconocido' #si no entiende el idioma devuelve desconocido

In [4]:
print(detectar_idioma("This hotel was amazing"))   # → 'en'
print(detectar_idioma("Das Hotel war sehr gut"))    # → 'de'
print(detectar_idioma("El hotel era fantástico"))   # → 'es'
print(detectar_idioma(""))                          # → 'es'
print(detectar_idioma(None))                        # → 'es'

en
de
es
es
es


In [5]:
def traducir_lote(textos):
    try:
        return GoogleTranslator(source='auto', target='es').translate_batch(textos) #le decimos coge el idioma que sea y me lo pasas al español y translate batch es para traducir por lotes
    except Exception as e:
        print(f"Error en lote: {e}")
        return textos  # si falla, devuelve los originales

# GESTIÓN DE LOS NAN

In [6]:
def gestionar_nulos(df, columnas):
    for col in columnas:
        df[col] = df[col].fillna('')
    df = df[~df[columnas].apply(lambda row: all(v == '' for v in row), axis=1)] # comprobación de si en una fila están todas las columnas vacías y nos quedamos con las que no son nan
    return df

## TRADUCCIÓN COMPLETA

In [7]:
def traduccion(columna, tamaño_lote=50, max_workers=10, checkpoint_path='checkpoint.csv'): # detecta el idioma y si no es español lo traduce
    print("  Detectando idiomas...")
    idiomas = columna.apply(detectar_idioma)
    
    mask_traducir = idiomas != 'es'
    indices_traducir = columna[mask_traducir].index
    textos_traducir = columna[mask_traducir].tolist()
    
    print(f"  Total: {len(columna)} | En español: {(~mask_traducir).sum()} | A traducir: {mask_traducir.sum()}")
    
    resultado = columna.copy()
    
    # Cargar checkpoint si existe (por si se cayó antes)
    if os.path.exists(checkpoint_path):
        print("  Cargando checkpoint previo...")
        checkpoint = pd.read_csv(checkpoint_path, index_col=0).squeeze()
        ya_hechos = checkpoint.index
        indices_traducir = indices_traducir.difference(ya_hechos)
        textos_traducir = columna[indices_traducir].tolist()
        resultado.update(checkpoint)
        print(f"  Ya traducidos: {len(ya_hechos)} | Quedan: {len(textos_traducir)}")
    
    # Crear lotes
    lotes = [
        (textos_traducir[i:i+tamaño_lote], indices_traducir[i:i+tamaño_lote])
        for i in range(0, len(textos_traducir), tamaño_lote)
    ]
    total_lotes = len(lotes)
    completados = 0

    # Traducir en paralelo
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futuros = {
            executor.submit(traducir_lote, lote): indices
            for lote, indices in lotes
        }
        
        for futuro in as_completed(futuros):
            indices_lote = futuros[futuro]
            try:
                traducidos = futuro.result()
                for idx, texto in zip(indices_lote, traducidos):
                    resultado[idx] = texto
            except Exception as e:
                print(f"  Error en lote: {e}")
            
            completados += 1

            # Guardar checkpoint cada 50 lotes
            if completados % 50 == 0:
                resultado[mask_traducir].to_csv(checkpoint_path)
                print(f"  {completados}/{total_lotes} lotes completados ({completados/total_lotes*100:.1f}%)")
    
    # Guardar checkpoint final
    resultado[mask_traducir].to_csv(checkpoint_path)
    print("Traducción completada!")
    return resultado

## EJECUCIÓN

In [8]:
df = pd.read_csv(r'C:\Users\esdia\Desktop\UV\NLP\PROYECTONLP\df_hoteles_vlc_comentarios.csv') #<-CAMBIAR EL DIRECTORIO
df

,nacionalidad,detalles_hab,noches,personas,comentario_general,positivo,negativo,nota,fecha,comentario_media_hotel
0,Colombia,NaN,2,familia,Espectacular,Espectacular todo! Restaurante; servicio; habi...,Todo me gustó,10.0,NaN,Fabuloso
1,España,NaN,1,pareja,Muy bien,NaN,NaN,9.0,NaN,Fabuloso
2,España,2 habitaciones Dobles Deluxe comunicadas,1,familia,Fantástico,"Habitaciones amplias, limpieza y ubicación",NaN,9.0,NaN,Fabuloso
3,España,NaN,1,pareja,Todo perfecto,El trato en recepción,El parking,10.0,NaN,Fabuloso
4,España,NaN,2,pareja,Una Nit del foc inolvidable.,Las vistas y el tamaño de la habitación.,"La calefacción central, sólo eso.",10.0,NaN,Fabuloso
...,...,...,...,...,...,...,...,...,...,...
58715,Alemania,NaN,1,pareja,Muy bien,NaN,NaN,8.0,NaN,Bien
58716,Italia,Habitación Doble Estándar con cama supletoria ...,2,familia,Excepcional,NaN,NaN,10.0,NaN,Bien
58717,España,Habitación Doble Estándar con cama supletoria ...,3,familia,Excepcional,NaN,NaN,10.0,NaN,Bien
58718,Rumanía,NaN,3,pareja,Perfect,NaN,NaN,10.0,NaN,Bien


In [9]:
# Gestionar NAN
df = gestionar_nulos(df, df.columns.tolist())
df

,nacionalidad,detalles_hab,noches,personas,comentario_general,positivo,negativo,nota,fecha,comentario_media_hotel
0,Colombia,,2,familia,Espectacular,Espectacular todo! Restaurante; servicio; habi...,Todo me gustó,10.0,,Fabuloso
1,España,,1,pareja,Muy bien,,,9.0,,Fabuloso
2,España,2 habitaciones Dobles Deluxe comunicadas,1,familia,Fantástico,"Habitaciones amplias, limpieza y ubicación",,9.0,,Fabuloso
3,España,,1,pareja,Todo perfecto,El trato en recepción,El parking,10.0,,Fabuloso
4,España,,2,pareja,Una Nit del foc inolvidable.,Las vistas y el tamaño de la habitación.,"La calefacción central, sólo eso.",10.0,,Fabuloso
...,...,...,...,...,...,...,...,...,...,...
58715,Alemania,,1,pareja,Muy bien,,,8.0,,Bien
58716,Italia,Habitación Doble Estándar con cama supletoria ...,2,familia,Excepcional,,,10.0,,Bien
58717,España,Habitación Doble Estándar con cama supletoria ...,3,familia,Excepcional,,,10.0,,Bien
58718,Rumanía,,3,pareja,Perfect,,,10.0,,Bien


In [ ]:
# Traducir al español
df['comentario_general'] = traduccion(df['comentario_general'])
df['positivo'] = traduccion(df['positivo'])
df['negativo'] = traduccion(df['negativo'])
df

  Detectando idiomas...
  Total: 58720 | En español: 23973 | A traducir: 34747
  50/695 lotes completados (7.2%)
  100/695 lotes completados (14.4%)
